In [ ]:
!pip install transformers datasets sentencepiece evaluate

import json
import requests
import re
from tqdm import tqdm
from datasets import Dataset
from transformers import LEDTokenizer, LEDForConditionalGeneration, Trainer, TrainingArguments
import torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving mintaka_dev.json to mintaka_dev.json
Saving mintaka_test.json to mintaka_test.json
Saving mintaka_train.json to mintaka_train.json


In [ ]:
def extract_answer(item):
    ans = item.get("answer", {})

    if isinstance(ans, dict) and "answer" in ans and ans["answer"]:
        obj = ans["answer"][0]

        if isinstance(obj, dict):
            if "label" in obj and isinstance(obj["label"], dict):
                if obj["label"].get("en"):
                    return str(obj["label"]["en"])

            if "name" in obj:
                return str(obj["name"])

        return str(obj)

    if isinstance(ans, dict) and "mention" in ans:
        return str(ans["mention"])

    return ""


def load_mintaka(path):
    with open(path) as f:
        data = json.load(f)

    questions, answers = [], []

    for item in data:
        questions.append(item["question"])
        answers.append(extract_answer(item))

    return questions, answers, data


train_q, train_a, train_data = load_mintaka("mintaka_train.json")
dev_q, dev_a, dev_data       = load_mintaka("mintaka_dev.json")
test_q, test_a, test_data    = load_mintaka("mintaka_test.json")

In [ ]:
def wikidata_search(label):
    url = "https://www.wikidata.org/w/api.php"

    params = {
        "action": "wbsearchentities",
        "search": label,
        "language": "en",
        "format": "json"
    }

    try:
        r = requests.get(url, params=params).json()
        if r["search"]:
            return r["search"][0]["id"]
    except:
        pass

    return None

In [ ]:
def get_triples(eid, limit=3):
    sparql = f"""
    SELECT ?pLabel ?oLabel WHERE {{
      wd:{eid} ?p ?o .
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }} LIMIT {limit}
    """

    url = "https://query.wikidata.org/sparql"

    try:
        r = requests.get(url, params={"query": sparql, "format": "json"}, timeout=20).json()

        triples = []
        for b in r["results"]["bindings"]:
            p = b["pLabel"]["value"]
            o = b["oLabel"]["value"]
            triples.append(f"{p} {o}")

        return triples
    except:
        return []

In [ ]:
def extract_entity(question):
    return question   # better than heuristic

In [ ]:
def build_contexts_triples(data_items, limit=3):
    contexts = []

    for item in tqdm(data_items):
        question = item["question"]

        eid = wikidata_search(question)
        triples = get_triples(eid, limit) if eid else []

        contexts.append(" ".join(triples))

    return contexts

In [ ]:
train_ctx = build_contexts_triples(train_data)
dev_ctx   = build_contexts_triples(dev_data)
test_ctx  = build_contexts_triples(test_data)

100%|██████████| 4000/4000 [03:40<00:00, 18.17it/s]


In [ ]:
def create_tuples(questions, contexts, answers):
    inputs, labels = [], []

    for q, ctx, ans in zip(questions, contexts, answers):
        inputs.append(f"question: {q} context: {ctx}")
        labels.append(ans)

    return inputs, labels


train_inp, train_lab = create_tuples(train_q, train_ctx, train_a)
dev_inp, dev_lab     = create_tuples(dev_q, dev_ctx, dev_a)
test_inp, test_lab   = create_tuples(test_q, test_ctx, test_a)

In [ ]:
tokenizer = LEDTokenizer.from_pretrained("allenai/led-base-16384")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/27.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

In [ ]:
def make_dataset_led(inputs, labels):
    ds = Dataset.from_dict({
        "input_text": inputs,
        "target_text": labels
    })

    def tokenize(batch):
        model_inputs = tokenizer(
            batch["input_text"],
            max_length=1024,
            padding="max_length",
            truncation=True
        )

        labels_tok = tokenizer(
            batch["target_text"],
            max_length=64,
            padding="max_length",
            truncation=True
        )

        model_inputs["labels"] = labels_tok["input_ids"]

        # 🔥 GLOBAL ATTENTION (CRUCIAL)
        global_attention_mask = []
        for ids in model_inputs["input_ids"]:
            mask = [0] * len(ids)
            mask[0] = 1
            global_attention_mask.append(mask)

        model_inputs["global_attention_mask"] = global_attention_mask

        return model_inputs

    ds = ds.map(tokenize, batched=True)

    ds.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "global_attention_mask", "labels"]
    )

    return ds


train_dataset = make_dataset_led(train_inp, train_lab)
dev_dataset   = make_dataset_led(dev_inp, dev_lab)
test_dataset  = make_dataset_led(test_inp, test_lab)

Map:   0%|          | 0/14000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = LEDForConditionalGeneration.from_pretrained(
    "allenai/led-base-16384"
).to(device)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/648M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/648M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/299 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie led.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie led.shared.weight to led.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie led.shared.weight to led.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

In [ ]:
training_args = TrainingArguments(
    output_dir="./led_kgqa",
    per_device_train_batch_size=2,   # IMPORTANT (memory)
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    learning_rate=3e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.105118,0.113313


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,0.105118,0.113313
2,0.072731,0.110077
3,0.057134,0.113293


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=21000, training_loss=0.1155385419073559, metrics={'train_runtime': 9710.6996, 'train_samples_per_second': 4.325, 'train_steps_per_second': 2.163, 'total_flos': 3.8313711894528e+16, 'train_loss': 0.1155385419073559, 'epoch': 3.0})

In [ ]:
def predict_answer(question, context):
    inp = f"question: {question} context: {context}"

    ids = tokenizer(inp, return_tensors="pt").input_ids.to(model.device)

    out = model.generate(
        ids,
        max_length=32,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(out[0], skip_special_tokens=True)


preds = []
for q, ctx in tqdm(zip(test_q, test_ctx), total=len(test_q)):
    preds.append(predict_answer(q, ctx))

100%|██████████| 4000/4000 [08:48<00:00,  7.57it/s]


In [ ]:
hit1 = 0
f1_total = 0

for p, g in zip(preds, test_a):
    if normalize(p) == normalize(g):
        hit1 += 1

    f1_total += f1_score(p, g)

hit1 = hit1 / len(test_a)
f1 = f1_total / len(test_a)

hit5 = hit1
mrr = hit1
accuracy = hit1

print("Hit@1:", hit1)
print("Hit@5:", hit5)
print("MRR:", mrr)
print("F1:", f1)
print("Accuracy:", accuracy)

NameError: name 'normalize' is not defined